# Eval harness from scratch

Interactive companion to `metrics.py` / `harness.py` / `run_smoke.py`.

Walk through: scoring functions → run 4 toy model variants → per-category breakdown → regression compare → release gate.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from dataset import EVAL_SET
from metrics import exact_match, token_f1, contains_match, rubric_grade
from models import make_models
from harness import run_eval, compare, gate

## 1. Scoring functions on one example

In [ ]:
ex = EVAL_SET[0]
golds = [ex['gold']] + ex['aliases']
for pred in ['Guido van Rossum', 'Python was created by Guido van Rossum.', 'James Gosling made Java.']:
    print(f'{pred!r:45s} EM={exact_match(pred, golds)} F1={token_f1(pred, golds):.2f} '
          f'contains={contains_match(pred, golds)} rubric={rubric_grade(pred, ex["rubric"])["score"]}')

## 2. Run every model variant

In [ ]:
models = make_models(seed=42)
runs = {name: run_eval(fn, EVAL_SET) for name, fn in models.items()}
for name, r in runs.items():
    print(f'{name:16s}', r['overall'])

## 3. Per-category breakdown

In [ ]:
for name, r in runs.items():
    print(name, {c: m['rubric_pass'] for c, m in r['per_category'].items()})

## 4. Regression comparison + gate

In [ ]:
cmp = compare(runs['rag_top1'], runs['rag_top3_concat'], max_drop=0.05)
print('deltas', cmp['overall_delta'])
print('regressions', cmp['regressions'], '| fail->pass', cmp['fail_to_pass'])
print(gate(runs['rag_top3_concat'], {'rubric_pass': 0.75, 'contains': 0.6}, cmp))